In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

## Đọc 2 file .csv chứa dữ liệu về review tích cực và tiêu cực

In [2]:
with open("/kaggle/input/datasets/meowmeowmindx/reviews-data/reviews-data/positive_data.csv", "r") as f:
    pos_df = pd.read_csv(f)
pos_df.head(20)

,Rate,Review,Label
0,9.0,Khu ẩm thực với đa dạng đồ lại còn bày trí đẹ...,1
1,9.0,Lúc nào đến aeon là lúc đấy phải tống một đốn...,1
2,10.0,Bánh ngon lại rẻ chê đâu được gần hết các loạ...,1
3,9.0,Ngon rẻ,1
4,9.6,Tôi sắp chết vì ngập trong sushi mấttttt Lên ...,1
5,9.0,Trong này rộng nên cứ đi lòng và lòng vòng mã...,1
6,10.0,Đến đây rất tiện vì cod xe bus rộng ăn tầng s...,1
7,9.0,Đến Aeon mall lần mặc dù mình đã có hẹn với b...,1
8,9.8,Nhắc đến Aeon Mình chỉ có thể nói đây là nơi ...,1
9,9.0,Đồ đồ uống đều phong đa dạng và Mình đi buổi ...,1


In [3]:
with open("/kaggle/input/datasets/meowmeowmindx/reviews-data/reviews-data/negative_data.csv", "r") as f:
    neg_df = pd.read_csv(f)
neg_df.head(20)

,Rate,Review,Label
0,4.0,Mình thề là mình ko thể cảm nổi đồ ăn ở aeon ...,-1
1,3.8,Đôi khi thèm lên là bất chấp nắng nóng phi Và...,-1
2,3.8,Ngõ treo biển cafe trứng đúng kiểu phố cổ hà ...,-1
3,3.8,Mình thấy địa chỉ cafe Giảng ở Nguyễn Hữu Huâ...,-1
4,2.2,Mình là người Hà Nội và cũng cực kỳ khó tính ...,-1
5,3.4,Mình là một người khá khó tính trong chuyện ă...,-1
6,3.8,Giá phù hợp sinh viên học cơ mà lần đầu tiên ...,-1
7,1.0,Mình đặt hàng cốc từ quán xác nhận đơn hàng m...,-1
8,1.0,Mếu tìm tkấy,-1
9,4.0,Hôm nay đi dạo qua nổi hứng muốn thử kem ốc q...,-1


In [4]:
# Sử dụng DataFrame.sample() để chọn số bình luận tích cực bằng tiêu cực
pos_df_sampled = pos_df.sample(n=len(neg_df), random_state=42)


In [5]:
# pd.concat() gộp hai dataframe vào thành một
df = pd.concat([pos_df_sampled, neg_df])
df['Label'].value_counts()

Label
 1    1765
-1    1765
Name: count, dtype: int64

In [6]:
# Sử dụng phương thức sample với frac=1 để: Xáo trộn dữ liệu 
# (lựa chọn 100% dữ liệu với thứ tự ngẫu nhiên)
df = df.sample(frac=1).reset_index(drop=True)
df.head()


,Rate,Review,Label
0,2.6,Không đắt phần vài miếng cá rán người nhưng t...,-1
1,10.0,Nằm trên tầng cao view nhìn toàn thành phố cự...,1
2,9.8,Quán chay đẹp và sang nhất mình từng Đồ ăn và...,1
3,9.2,hôm trc vừa đi vs người đang tán bình thường ...,1
4,2.8,Mình là khách hàng thường xuyên ăn ở đây do t...,-1


In [7]:
# DataFrame.replace() nhận vào một dictionary
# với key là giá trị cần thay đổi và value là giá trị mới.
df['Label'] = df['Label'].replace({-1:0})
df['Label'].value_counts()
# => Thay thế các giá trị -1 trong cột Label thành giá trị 0.
# Vì đầu ra của mô hình là các giá trị có khoảng từ 0 - 1 
# Gần với 0 => tiêu cực, gần với 1 => tích cực.

Label
0    1765
1    1765
Name: count, dtype: int64

## Chia tập train - test

In [8]:
# Chia tỉ lệ train:test là. 8:2
training_size = int(len(df)*0.8)

train = df.iloc[:training_size]
train_text = train['Review']
train_labels = train["Label"].to_numpy()

test = df.iloc[training_size:]
test_text = test['Review']
test_labels = test["Label"].to_numpy()

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 2824
Test size: 706


## Tokenize

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


2026-05-17 07:12:30.956506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779001951.224651      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779001951.305055      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779001951.960180      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779001951.960234      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779001951.960238      57 computation_placer.cc:177] computation placer alr

In [10]:
# Chọn kích thước từ điển là: 10000
# Và độ dài lớn nhất của một bình luận là: 2000
vocab_size = 10000
max_length = 2000

# Tạo Tokenizer và fit trên tập train
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

# Sử dụng Tokenizer đã fit để tokenize các bình luận trên tập train
train_seqs = tokenizer.texts_to_sequences(train_text)
train_paded = pad_sequences(train_seqs, maxlen=max_length, padding='post',truncating='post')

# Sử dụng Tokenizer đã fit để tokenize các bình luận trên tập test
test_seqs = tokenizer.texts_to_sequences(test_text)
test_paded = pad_sequences(test_seqs, maxlen=max_length, padding='post',truncating='post')

## Định nghĩa tokenizer và tokenzize tập train - test trước khi huấn luyện. 

## Định nghĩa mô hình

Kiến trúc mô hình tham khảo, gồm 1 layer embedding và 1 block phân loại

In [11]:
from tensorflow import keras

# chọn chiều embedding là 128
embedding_dim = 128

model = keras.Sequential([
    # Layer Embedding
    keras.layers.Embedding(vocab_size, embedding_dim, trainable=True), 
    # vocab_size=10000, max_length=2000

    # Layer phân loại (đã học)
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
    
])
model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])
# mô hình phân loại thường sử dụng thang đo là accuracy

model.summary()

2026-05-17 07:13:00.339206: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Huấn luyện mô hình

In [12]:
num_epochs = 10
model.fit(train_paded,
          train_labels, 
          epochs=num_epochs, 
          shuffle=True, 
          validation_data=[test_paded,test_labels])

Epoch 1/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 24s 245ms/step - accuracy: 0.5047 - loss: 8.3247 - val_accuracy: 0.4788 - val_loss: 32.6979
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - accuracy: 0.5692 - loss: 13.0787 - val_accuracy: 0.7918 - val_loss: 0.5156
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 22s 252ms/step - accuracy: 0.8902 - loss: 0.2686 - val_accuracy: 0.8229 - val_loss: 0.3938
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 22s 246ms/step - accuracy: 0.9413 - loss: 0.1645 - val_accuracy: 0.8513 - val_loss: 0.3473
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 22s 249ms/step - accuracy: 0.9800 - loss: 0.0910 - val_accuracy: 0.8470 - val_loss: 0.3503
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 22s 251ms/step - accuracy: 0.9887 - loss: 0.0595 - val_accuracy: 0.8626 - val_loss: 0.3490
Epoch 7/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 21s 239ms/step - accuracy: 0.9919 - loss: 0.0451 - val_accuracy: 0.8598 - val_loss: 0.3546
Epoch 8/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 22s 244ms/step - accuracy: 0.9962 - loss: 0.0297 - val_ac

In [13]:
# Run xong hết epochs thì mới chạy dòng này
model.save("model.keras") # lưu mô hình với phương thức .save()

## Sử dụng mô hình

In [14]:
# Tạo một số dữ liệu mẫu để đánh giá mô hình
test_reviews = [
    "Thịt bị hôi. Nước bún nhạt. Đề nghị dừng hoạt động",
    "Rất thích không gian quán",
    "Tôi không thích món dưa leo ở đây chút nào",
    "Nhiều đồ ăn, khá ngon. Sẽ quay lại lần sau nhưng giá cả hơi mắc.",
    "Nhân viên phục còn chưa được chu đáo"
]

test_gt = [0, 1, 0, 1, 0]
test_df = pd.DataFrame({"Review": test_reviews, "Label": test_gt})
test_df

,Review,Label
0,Thịt bị hôi. Nước bún nhạt. Đề nghị dừng hoạt ...,0
1,Rất thích không gian quán,1
2,Tôi không thích món dưa leo ở đây chút nào,0
3,"Nhiều đồ ăn, khá ngon. Sẽ quay lại lần sau như...",1
4,Nhân viên phục còn chưa được chu đáo,0


## Các bước sử dụng mô hình để phân loại: 
1. Sử dụng tokenizer đã fit ở bước trước để tokenize các bình luận
2. Thực hiện padding tương ứng
3. Dự đoán bằng phương thức predict với mô hình
4. Xử lý kết quả dự đoán

In [15]:
# Sử dụng lại Tokenizer
test_reviews = test_df["Review"].values
test_labels = test_df["Label"].values

# Thực hiện tokenizer cái nào? 
test_seqs = tokenizer.texts_to_sequences(test_reviews)
# Thực hiện padding tương ứng
test_padded = pad_sequences(test_seqs, padding="post", truncating='post', maxlen=max_length)

## Dự đoán với mô hình

In [16]:
from tensorflow.keras.models import load_model

loaded_model = load_model("/kaggle/working/model.keras")
preds = loaded_model.predict(test_padded)
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


array([[0.01381611],
       [0.99842584],
       [0.95606667],
       [0.802353  ],
       [0.71249443]], dtype=float32)

# So sánh kết quả dự đoán của mô hình với nhãn thực tế trên tập kiểm tra (test_reviews).

In [20]:
classes = {
    "1":"positive",
    "0": "negative"
} # chuyển đổi nhãn 1, 0 => ngôn ngữ tự nhiên dễ đọc.
acc = 0 # Biến đếm số lượng dự đoán đúng (Accuracy counter)
i = 0   # Biến chỉ số => theo dõi vị trí của phần tử hiện tại trong danh sách nhãn thực tế.
threshold = 0.5  # Ngưỡng phân loại. Nếu điểm số lớn hơn 0.5, mô hình sẽ coi đó là tích cực (1),
# Ngược lại là tiêu cực (0)

In [21]:
# Gộp từng câu nhận xét(fb) với điểm số dự đoán tương ứng (score) để xử lý song song mỗi vòng lặp
for fb, score in zip(test_reviews, preds): 
    # So sánh điểm số với ngưỡng 0.5. (True) => chuyển thành 1, Nếu sai (False) => chuyển thành 0
    # Định dạng: astype(int) -> dạng mảng NumPy
    prediction = (score>threshold).astype(int)
    print(fb, "|", classes[str(test_labels[i])], "predicted as", 
          classes[str(prediction[0])], "with score =", score[0])
    
    if prediction == test_labels[i]:
        acc += 1
    i += 1

print("\nAccuracy:", round(acc/len(test_reviews), 2))



Thịt bị hôi. Nước bún nhạt. Đề nghị dừng hoạt động | negative predicted as negative with score = 0.013816114
Rất thích không gian quán | positive predicted as positive with score = 0.99842584
Tôi không thích món dưa leo ở đây chút nào | negative predicted as positive with score = 0.95606667
Nhiều đồ ăn, khá ngon. Sẽ quay lại lần sau nhưng giá cả hơi mắc. | positive predicted as positive with score = 0.802353
Nhân viên phục còn chưa được chu đáo | negative predicted as positive with score = 0.71249443

Accuracy: 0.6
